In [1]:
# 
# Get holdings for a fund a given preiod of report (financial quarter).
# A fund can file an initial form and subsequent amendments for a period of report .
# This script aggregates all such forms inti a final holding set.
#

In [1]:
# install forms13f python sdk 
try:
    import forms13f 
    print('forms13f forms13f package is already installed')
except ImportError:
    # Package not found, so install it from GitHub
    !pip install git+https://github.com/forms13f/python-sdk.git


forms13f forms13f package is already installed


In [2]:
# Get aggregated quarter report
from pprint import pprint
from report import *
# Example usage
quarter = "2023-Q3"
cik = "0001067983"  # Example CIK for Berkshire Hathaway Inc
quarter_report = getQuarterReport(quarter, cik)
pprint(quarter_report.header.__dict__)
pprint(quarter_report.holdings)

{'accession_numbers': ['0000950123-23-010898',
                       '0000950123-23-011029',
                       '0000950123-24-005653'],
 'business_address': '3555 FARNAM STREET, OMAHA, NE, 68131',
 'business_phone': '4023461400',
 'cik': '0001067983',
 'company_name': 'BERKSHIRE HATHAWAY INC',
 'date_as_of_change': datetime.date(2023, 11, 14),
 'effectiveness_date': datetime.date(2023, 11, 14),
 'filing_dates': [datetime.date(2023, 11, 14),
                  datetime.date(2023, 11, 16),
                  datetime.date(2024, 5, 15)],
 'fiscal_year_end': '1231',
 'form_type': '13F-HR',
 'irs_number': '470813844',
 'period_of_report': datetime.date(2023, 9, 30),
 'public_document_count': 2,
 'report_quarter': '2023-Q3',
 'sec_act': '1934 Act',
 'state_of_incorporation': 'DE',
 'submission_type': '13F-HR',
 'table_entry_total': 152,
 'table_value_total': 313257308189,
 'urls': ['https://www.sec.gov/Archives/edgar/data/1067983/0000950123-23-010898.txt',
          'https://www.sec.gov/

In [3]:
from pprint import pprint
from report import *

# Define the quarters to iterate over
quarters = ['2023-Q1', '2023-Q2', '2023-Q3', '2024-Q1', '2024-Q2']

cik = "0001067983"  # Example CIK for Berkshire Hathaway Inc

# Initialize the reports array
reports = []

# Iterate over the quarters and append the reports to the array
for quarter in quarters:
    quarter_report = getQuarterReport(quarter, cik)
    reports.append(quarter_report)


In [11]:
import pandas as pd
from datetime import date
import matplotlib.pyplot as plt

# List to hold all report data
data = []

# Loop through each QuarterReport to collect data
for report in reports:
    for holding in report.holdings:
        header = report.header
        data.append({
            'cik': header.cik,
            'company_name': header.company_name,
            'period_of_report': header.period_of_report,
            'report_quarter': header.report_quarter,  # Assuming report_quarter is like '2024-Q3'
            'name_of_issuer': holding.name_of_issuer,
            'title_of_class': holding.title_of_class,
            'cusip': holding.cusip,
            'ticker': holding.ticker,
            'value': holding.value,
            'ssh_prnamt': holding.ssh_prnamt,
            'ssh_prnamt_type': holding.ssh_prnamt_type,
        })



In [17]:
import pandas as pd
import matplotlib.pyplot as plt

# Assuming `data` has already been populated and the DataFrame is set up as before
df = pd.DataFrame(data)

df.set_index(['cik', 'cusip', 'report_quarter'], inplace=True)

# Calculate ssh_prnamt as a fraction of the first non-zero value for each holding in its reporting period
def first_non_zero(series):
    non_zero_values = series[series != 0]
    return non_zero_values.iloc[0] if not non_zero_values.empty else 1  # Use 1 to avoid division by zero

# Calculate the baseline percentage relative to the first non-zero value
earliest_ssh_prnamt = df.groupby(level=['cik', 'cusip'])['ssh_prnamt'].transform(first_non_zero)
df['ssh_prnamt_percent'] = (df['ssh_prnamt'] / earliest_ssh_prnamt) * 100
print(df[['ticker', 'period_of_report', 'ssh_prnamt_percent']])



                                        ticker period_of_report  \
cik        cusip     report_quarter                               
0001067983 00507V109 2023-Q1         00507V109       2023-03-31   
           02005N100 2023-Q1              ALLY       2023-03-31   
           023135106 2023-Q1              AMZN       2023-03-31   
           025816109 2023-Q1               AXP       2023-03-31   
           G0403H108 2023-Q1               AON       2023-03-31   
...                                        ...              ...   
           872590104 2024-Q2              TMUS       2024-06-30   
           90384S303 2024-Q2              ULTA       2024-06-30   
           922908363 2024-Q2               VOO       2024-06-30   
           92343E102 2024-Q2              VRSN       2024-06-30   
           92826C839 2024-Q2                 V       2024-06-30   

                                     ssh_prnamt_percent  
cik        cusip     report_quarter                      
0001067983 0